# Databricks notebook source
##### Gold Layer Data Load: Product Dimension (`gold_dim_products`)

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

##### 1. Initialize Gold Schema & Target Table Name

In [0]:
catalog_name = "glrecon"
gold_table_name = f"{catalog_name}.gold.gold_dim_products"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.gold")

###### 2. Read & Transform Silver Product Dimension

In [0]:
df_silver_prod = spark.table(f"{catalog_name}.silver.silver_dim_products").select(
    F.col("product_id").cast("long").alias("ProductId"),
    F.col("product_name").alias("ProductName"),
    F.col("category").alias("Category"),
    F.col("is_active").alias("IsActive"),
    F.col("create_date").alias("CreateDate"),
    F.current_timestamp().alias("GoldUpdatedDate")
)

###### 3. Upsert into Gold Product Dimension (`MERGE`)

In [0]:

if not spark.catalog.tableExists(gold_table_name):
    print(f"Table {gold_table_name} does not exist. Initializing Gold dimension table...")
    
    df_silver_prod.write.format("delta").mode("overwrite").saveAsTable(gold_table_name)
    print("Gold product dimension successfully created and populated.")

else:
    print(f"Table {gold_table_name} exists. Proceeding with MERGE pipeline...")
    
    gold_table = DeltaTable.forName(spark, gold_table_name)

    (
        gold_table.alias("target")
        .merge(
            df_silver_prod.alias("source"),
            "target.ProductId = source.ProductId"
        )
        .whenMatchedUpdate(
            condition="""
                NOT (target.ProductName <=> source.ProductName) OR
                NOT (target.Category <=> source.Category) OR
                target.IsActive <> source.IsActive
            """,
            set={
                "ProductName": "source.ProductName",
                "Category": "source.Category",
                "IsActive": "source.IsActive",
                "GoldUpdatedDate": F.current_timestamp()
            }
        )
        .whenNotMatchedInsert(
            values={
                "ProductId": "source.ProductId",
                "ProductName": "source.ProductName",
                "Category": "source.Category",
                "IsActive": "source.IsActive",
                "CreateDate": "source.CreateDate",
                "GoldUpdatedDate": F.current_timestamp()
            }
        )
        .execute()
    )
    print("Merge execution completed successfully.")




####### Read the data from Gold Product table

In [0]:
%sql
SELECT * FROM glrecon.gold.gold_dim_products